# Motility to Cell Painting Pipeline Starter

This notebook sketches the handoff from raw experiment folders to the aligner widget:

1. discover live-cell TIFF frames and Cell Painting tiles
2. run Cellpose on the live-cell frames
3. convert Cellpose masks into per-frame object tables
4. link objects into live-cell tracks
5. run or load CellProfiler object measurements for Cell Painting tiles
6. use the terminal live-cell frame in the aligner widget
7. export curated terminal live-cell to Cell Painting matches

Keep the first pass small: run one experiment date, inspect QC images, tune thresholds, then batch it.

In [ ]:
from pathlib import Path

import pandas as pd

from cell_motility_painting_aligner import MotilityPaintingAligner
from cell_motility_painting_aligner.pipeline import (
    build_cellpose_command,
    build_cellprofiler_command,
    candidate_matches_from_transform,
    discover_cell_painting_tiles,
    discover_live_cell_frames,
    link_objects_by_centroid,
    merge_curated_matches,
    objects_from_cellpose_segmentation,
    terminal_detections,
    terminal_live_cell_frame,
    track_feature_table,
    write_cellprofiler_file_list,
)


## Configure experiment paths

In [ ]:
experiment_date = "2026-05-10"

data_root = Path("../data")
output_root = Path("../outputs")

cell_painting_dir = data_root / "cell_painting" / experiment_date
live_tiff_dir = data_root / "live_cell_imaging" / experiment_date / "tiff files"

live_output_dir = output_root / "live_cell_imaging" / experiment_date
cellpose_output_dir = live_output_dir / "cellpose_masks"
cell_painting_output_dir = output_root / "cell_painting" / experiment_date
matches_output_dir = output_root / "matches" / experiment_date

for path in [cellpose_output_dir, cell_painting_output_dir, matches_output_dir]:
    path.mkdir(parents=True, exist_ok=True)


## Discover input files

In [ ]:
live_frames = discover_live_cell_frames(live_tiff_dir)
cell_painting_tiles = discover_cell_painting_tiles(cell_painting_dir)
last_frame = terminal_live_cell_frame(live_frames)

print(f"live frames: {len(live_frames)}")
print(f"terminal frame: Time{last_frame.time_index:05d} -> {last_frame.path.name}")
print(f"cell painting tiles: {len(cell_painting_tiles)}")
cell_painting_tiles[:3]


## Build and run Cellpose command

The command is shown first so you can inspect the model, channel settings, diameter, and output directory. Uncomment the run cell once the command looks right.

In [ ]:
cellpose_cmd = build_cellpose_command(
    live_tiff_dir,
    output_dir=cellpose_output_dir,
    model="cyto2",
    diameter=None,
    chan=0,
    chan2=0,
    use_gpu=False,
    save_tif=True,
    save_outlines=True,
)
cellpose_cmd


In [ ]:
# from cell_motility_painting_aligner.pipeline import run_cellpose_cli
# run_cellpose_cli(
#     live_tiff_dir,
#     output_dir=cellpose_output_dir,
#     model="cyto2",
#     diameter=None,
#     chan=0,
#     chan2=0,
#     use_gpu=False,
#     save_tif=True,
#     save_outlines=True,
# )


## Convert Cellpose segmentations into object tables

In [ ]:
seg_paths = sorted(cellpose_output_dir.glob("*_seg.npy"))
frame_by_name = {frame.path.stem: frame for frame in live_frames}

live_objects = []
for seg_path in seg_paths:
    image_stem = seg_path.stem.replace("_seg", "")
    frame = frame_by_name.get(image_stem)
    frame_index = frame.time_index if frame is not None else None
    live_objects.extend(
        objects_from_cellpose_segmentation(
            seg_path,
            image_id=image_stem,
            frame=frame_index,
            min_area=25,
        )
    )

live_objects_df = pd.DataFrame(live_objects)
live_objects_path = live_output_dir / "per_frame_objects.parquet"
if not live_objects_df.empty:
    live_objects_df.to_parquet(live_objects_path, index=False)
live_objects_df.head()


## Track cells frame to frame

In [ ]:
if live_objects_df.empty:
    tracked = []
    tracks_df = pd.DataFrame()
    terminal_df = pd.DataFrame(columns=["track_id", "object_id", "frame", "x", "y"])
    motility_features_df = pd.DataFrame()
    print("No Cellpose object rows found yet. Run Cellpose, then rerun this section.")
else:
    tracked = link_objects_by_centroid(
        live_objects_df.to_dict("records"),
        max_distance=35,
        max_frame_gap=1,
    )
    tracks_df = pd.DataFrame(tracked)
    terminal_df = pd.DataFrame(terminal_detections(tracked))
    motility_features_df = pd.DataFrame(track_feature_table(tracked))

    tracks_df.to_parquet(live_output_dir / "tracks.parquet", index=False)
    terminal_df.to_parquet(live_output_dir / "terminal_objects.parquet", index=False)
    motility_features_df.to_parquet(live_output_dir / "motility_features.parquet", index=False)

motility_features_df.head()


## Build CellProfiler command for Cell Painting tiles

Create and tune the `.cppipe` pipeline in CellProfiler first, then use the command below for batch execution.

In [ ]:
cellprofiler_pipeline = Path("../pipelines/cell_painting.cppipe")
file_list_path = write_cellprofiler_file_list(
    [tile.path for tile in cell_painting_tiles],
    cell_painting_output_dir / "cellprofiler_file_list.txt",
)
cellprofiler_cmd = build_cellprofiler_command(
    cellprofiler_pipeline,
    cell_painting_output_dir,
    file_list=file_list_path,
)
cellprofiler_cmd


In [ ]:
# from cell_motility_painting_aligner.pipeline import run_cellprofiler_cli
# run_cellprofiler_cli(
#     cellprofiler_pipeline,
#     cell_painting_output_dir,
#     file_list=file_list_path,
# )


## Load Cell Painting object centroids

Normalize the CellProfiler output to at least these columns: `object_id`, `tile_id`, `x`, `y`. Keep the full feature table for the final phenotype join.

In [ ]:
cell_painting_objects_path = cell_painting_output_dir / "cellprofiler_objects.parquet"
# cell_painting_objects_df = pd.read_parquet(cell_painting_objects_path)

# Placeholder schema for development before CellProfiler output is available.
cell_painting_objects_df = pd.DataFrame(
    columns=["object_id", "tile_id", "image_index", "x", "y"]
)
cell_painting_objects_df.head()


## Launch the aligner on the terminal live-cell frame

The widget needs display images, so point these paths at PNG/JPEG renderings. The raw TIFF/IMS files still remain the source for segmentation and features.

In [ ]:
terminal_rendered_path = live_output_dir / "rendered_frames" / f"Time{last_frame.time_index:05d}.png"
rendered_tile_paths = [
    cell_painting_output_dir / "rendered_tiles" / f"{tile.tile_id}.png"
    for tile in cell_painting_tiles
]

terminal_centroids = (
    terminal_df.assign(id=terminal_df["track_id"].astype(str))[["id", "x", "y"]]
    if not terminal_df.empty
    else []
)

cell_painting_centroids_by_image = []
for image_index, tile in enumerate(cell_painting_tiles):
    tile_objects = cell_painting_objects_df[cell_painting_objects_df["tile_id"] == tile.tile_id]
    tile_centroids = tile_objects.assign(id=tile_objects["object_id"].astype(str))[["id", "x", "y"]]
    cell_painting_centroids_by_image.append(tile_centroids.to_dict("records"))

missing_review_images = [path for path in [terminal_rendered_path, *rendered_tile_paths] if not path.exists()]

if missing_review_images:
    w = None
    print("Render terminal live-cell and Cell Painting review PNGs before launching the widget.")
    print(f"Missing {len(missing_review_images)} review image(s). First missing path:")
    print(missing_review_images[0])
else:
    # Fill in real image sizes after rendering.
    w = MotilityPaintingAligner.from_paths(
        terminal_rendered_path,
        rendered_tile_paths,
        motility_size=[2048, 2048],
        cell_painting_size=[2048, 2048],
        motility_centroids=(
            terminal_centroids.to_dict("records")
            if hasattr(terminal_centroids, "to_dict")
            else terminal_centroids
        ),
        cell_painting_centroids_by_image=cell_painting_centroids_by_image,
    )

w


## Export curated matches and join phenotypes

In [ ]:
# After manual review in the widget:
# fit = w.fit()
match_rows = w.export_matches() if w is not None else []
matches_df = pd.DataFrame(match_rows)
matches_df.to_csv(matches_output_dir / "terminal_live_to_cell_painting_matches.csv", index=False)

joined_matches = merge_curated_matches(
    match_rows,
    terminal_df.to_dict("records"),
    cell_painting_objects_df.to_dict("records"),
)
joined_matches_df = pd.DataFrame(joined_matches)
joined_matches_df.head()


## Optional: propose candidates from a fitted transform

If a tile has enough landmark pairs, the aligner transform can project Cell Painting centroids into live-cell final-frame coordinates and suggest nearby candidates.

In [ ]:
image_index = 0
transform = w.transform_by_image.get(str(image_index)) if w is not None else None
if transform:
    tile_id = cell_painting_tiles[image_index].tile_id
    tile_objects = cell_painting_objects_df[cell_painting_objects_df["tile_id"] == tile_id]
    candidates = candidate_matches_from_transform(
        terminal_df.to_dict("records"),
        tile_objects.to_dict("records"),
        transform,
        max_distance=25,
        max_candidates=3,
    )
    candidate_df = pd.DataFrame(candidates)
else:
    candidate_df = pd.DataFrame()

candidate_df.head()


## Join motility and Cell Painting feature tables

Once the curated match table is stable, join the terminal match IDs to `motility_features.parquet` and the full CellProfiler feature table.

In [ ]:
# cell_painting_features_df = pd.read_parquet(cell_painting_output_dir / "cellprofiler_features.parquet")
# analysis_df = (
#     matches_df
#     .merge(motility_features_df, left_on="motility_id", right_on="track_id", how="left")
#     .merge(cell_painting_features_df, left_on="cell_painting_id", right_on="object_id", how="left")
# )
# analysis_df.to_parquet(
#     output_root / "analysis" / experiment_date / "joined_motility_cell_painting_features.parquet",
#     index=False,
# )
